# Veritrix — Customer Support Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/veritrix-ai/veritrix-sdk/blob/master/examples/customer_support_agent.ipynb)

Airline customer-support demo with the [OpenAI Agents SDK](https://github.com/openai/openai-agents-python):
triage, FAQ, and seat-booking agents with tool calls and handoffs.

Each turn is wrapped in `veritrix.trace()` so you can inspect the full multi-agent run in the Veritrix dashboard.

**What you need**
1. A Veritrix API key
2. An OpenAI API key (`OPENAI_API_KEY`)

**Local backend tip:** Colab cannot reach `localhost`. Tunnel ingest with `ngrok http 8001` and set `VERITRIX_ENDPOINT` to the public HTTPS URL.

## 1. Install dependencies

In [ ]:
!pip install -q "git+https://github.com/veritrix-ai/veritrix-sdk.git" openai-agents pydantic

## 2. Configure keys

In [ ]:
import os

VERITRIX_API_KEY = ""  # @param {type:"string"}
VERITRIX_ENDPOINT = "http://localhost:8001"  # @param {type:"string"}
OPENAI_API_KEY = ""  # @param {type:"string"}

if not VERITRIX_API_KEY.strip():
    raise ValueError("Set VERITRIX_API_KEY before continuing.")
if not OPENAI_API_KEY.strip():
    raise ValueError("Set OPENAI_API_KEY before continuing.")

os.environ["VERITRIX_API_KEY"] = VERITRIX_API_KEY
os.environ["VERITRIX_ENDPOINT"] = VERITRIX_ENDPOINT
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

## 3. Define agents, tools, and handoffs

- **Triage Agent** — routes the customer to FAQ or seat booking
- **FAQ Agent** — answers baggage / seats / wifi questions via a lookup tool
- **Seat Booking Agent** — updates a seat with confirmation number + tool call

In [ ]:
from __future__ import annotations

import random
import uuid

from pydantic import BaseModel

from agents import (
    Agent,
    HandoffOutputItem,
    ItemHelpers,
    MessageOutputItem,
    RunContextWrapper,
    Runner,
    ToolCallItem,
    ToolCallOutputItem,
    TResponseInputItem,
    function_tool,
    handoff,
    trace as agents_trace,
)
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX

DEMO_MESSAGES = [
    "What is the baggage allowance for my flight?",
    "I want to change my seat to 12A. My confirmation number is ABC123.",
]


class AirlineAgentContext(BaseModel):
    passenger_name: str | None = None
    confirmation_number: str | None = None
    seat_number: str | None = None
    flight_number: str | None = None


@function_tool(
    name_override="faq_lookup_tool",
    description_override="Lookup frequently asked questions.",
)
async def faq_lookup_tool(question: str) -> str:
    q = question.lower()
    if "bag" in q or "baggage" in q:
        return (
            "You are allowed to bring one bag on the plane. "
            "It must be under 50 pounds and 22 inches x 14 inches x 9 inches."
        )
    if "seats" in q or "plane" in q:
        return (
            "There are 120 seats on the plane. "
            "There are 22 business class seats and 98 economy seats. "
            "Exit rows are rows 4 and 16. "
            "Rows 5-8 are Economy Plus, with extra legroom."
        )
    if "wifi" in q:
        return "We have free wifi on the plane, join Airline-Wifi"
    return "I'm sorry, I don't know the answer to that question."


@function_tool
async def update_seat(
    context: RunContextWrapper[AirlineAgentContext],
    confirmation_number: str,
    new_seat: str,
) -> str:
    context.context.confirmation_number = confirmation_number
    context.context.seat_number = new_seat
    assert context.context.flight_number is not None, "Flight number is required"
    return f"Updated seat to {new_seat} for confirmation number {confirmation_number}"


async def on_seat_booking_handoff(context: RunContextWrapper[AirlineAgentContext]) -> None:
    context.context.flight_number = f"FLT-{random.randint(100, 999)}"


def build_agents() -> Agent[AirlineAgentContext]:
    faq_agent = Agent[AirlineAgentContext](
        name="FAQ Agent",
        handoff_description="A helpful agent that can answer questions about the airline.",
        instructions=f"""{RECOMMENDED_PROMPT_PREFIX}
        You are an FAQ agent. If you are speaking to a customer, you probably were transferred from the triage agent.
        Use the following routine to support the customer.
        # Routine
        1. Identify the last question asked by the customer.
        2. Use the faq lookup tool to answer the question. Do not rely on your own knowledge.
        3. If you cannot answer the question, transfer back to the triage agent.""",
        tools=[faq_lookup_tool],
    )

    seat_booking_agent = Agent[AirlineAgentContext](
        name="Seat Booking Agent",
        handoff_description="A helpful agent that can update a seat on a flight.",
        instructions=f"""{RECOMMENDED_PROMPT_PREFIX}
        You are a seat booking agent. If you are speaking to a customer, you probably were transferred from the triage agent.
        Use the following routine to support the customer.
        # Routine
        1. Ask for their confirmation number.
        2. Ask the customer what their desired seat number is.
        3. Use the update seat tool to update the seat on the flight.
        If the customer asks a question that is not related to the routine, transfer back to the triage agent.""",
        tools=[update_seat],
    )

    triage_agent = Agent[AirlineAgentContext](
        name="Triage Agent",
        handoff_description="A triage agent that can delegate a customer's request to the appropriate agent.",
        instructions=(
            f"{RECOMMENDED_PROMPT_PREFIX} "
            "You are a helpful triaging agent. You can use your tools to delegate questions to other appropriate agents."
        ),
        handoffs=[
            faq_agent,
            handoff(agent=seat_booking_agent, on_handoff=on_seat_booking_handoff),
        ],
    )

    faq_agent.handoffs.append(triage_agent)
    seat_booking_agent.handoffs.append(triage_agent)
    return triage_agent


def print_agent_output(new_items: list[object]) -> None:
    for new_item in new_items:
        agent_name = new_item.agent.name  # type: ignore[attr-defined]
        if isinstance(new_item, MessageOutputItem):
            print(f"{agent_name}: {ItemHelpers.text_message_output(new_item)}")
        elif isinstance(new_item, HandoffOutputItem):
            print(f"Handed off from {new_item.source_agent.name} to {new_item.target_agent.name}")
        elif isinstance(new_item, ToolCallItem):
            print(f"{agent_name}: Calling a tool")
        elif isinstance(new_item, ToolCallOutputItem):
            print(f"{agent_name}: Tool call output: {new_item.output}")
        else:
            print(f"{agent_name}: Skipping item: {new_item.__class__.__name__}")


print("Agents and tools ready.")

## 4. Run the demo conversation

Sends two scripted customer messages:
1. Baggage allowance → FAQ path
2. Seat change → seat-booking path + tool call

In [ ]:
import veritrix
from veritrix.config import get_config


async def run_turn(
    *,
    current_agent: Agent[AirlineAgentContext],
    input_items: list[TResponseInputItem],
    context: AirlineAgentContext,
    conversation_id: str,
    user_input: str,
    turn_index: int,
) -> tuple[Agent[AirlineAgentContext], list[TResponseInputItem]]:
    input_items.append({"content": user_input, "role": "user"})

    with veritrix.trace(
        f"turn-{turn_index}",
        span_type="agent",
        input_data={"message": user_input, "conversation_id": conversation_id},
    ):
        with agents_trace("Customer service", group_id=conversation_id):
            result = await Runner.run(current_agent, input_items, context=context)

    print_agent_output(result.new_items)
    return result.last_agent, result.to_input_list()


veritrix.init(
    api_key=VERITRIX_API_KEY,
    endpoint=VERITRIX_ENDPOINT,
    default_tags=["customer-service-agent", "openai-agents", "colab"],
    framework="manual",
    agent_name="Customer Service",
)

trace_id = get_config().trace_id
print(f"Veritrix trace_id: {trace_id}")
print(f"Ingest endpoint: {VERITRIX_ENDPOINT}\n")

current_agent = build_agents()
input_items: list[TResponseInputItem] = []
context = AirlineAgentContext()
conversation_id = uuid.uuid4().hex[:16]

try:
    for turn_index, message in enumerate(DEMO_MESSAGES, start=1):
        print(f"You: {message}")
        current_agent, input_items = await run_turn(
            current_agent=current_agent,
            input_items=input_items,
            context=context,
            conversation_id=conversation_id,
            user_input=message,
            turn_index=turn_index,
        )
        print()
finally:
    print("Flushing spans to Veritrix...")
    veritrix.end()

print(f"\nDone. Open Traces in Veritrix and look for trace_id={trace_id}")